# AI Code Suggester — QLoRA Fine-Tuning

**Runtime:** T4 GPU (free tier)  
**Model:** `Salesforce/codet5-base`  
**Method:** QLoRA (4-bit quantization + LoRA adapters)

## Steps
1. Install dependencies
2. Mount Google Drive
3. Upload dataset
4. Train
5. Save weights to Drive
6. Evaluate

In [ ]:
# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — switch runtime to T4!')

In [ ]:
!pip uninstall -y bitsandbytes 2>/dev/null; true
!pip install -q \
    transformers==4.40.0 \
    peft==0.10.0 \
    accelerate==0.29.3 \
    datasets==2.19.0 \
    sentencepiece scipy tqdm
print('\n✅ Install complete.')
print('👉 Runtime → Restart session, then run all cells again.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ai-code-suggester'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print('Drive mounted. Working dir:', DRIVE_DIR)

In [ ]:
# Option A: Upload dataset files from your computer
from google.colab import files
print('Upload dataset_train.jsonl and dataset_eval.jsonl')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = f'{DRIVE_DIR}/data/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'Saved {fname} → {dest}')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
MODEL_NAME       = 'Salesforce/codet5-base'
TRAIN_FILE       = f'{DRIVE_DIR}/data/dataset_train.jsonl'
EVAL_FILE        = f'{DRIVE_DIR}/data/dataset_eval.jsonl'
OUTPUT_DIR       = f'{DRIVE_DIR}/checkpoints'
MAX_INPUT_LEN    = 256
MAX_TARGET_LEN   = 128
EPOCHS           = 3
BATCH_SIZE       = 8   # T4 can handle 8 for codet5-base
LORA_R           = 16
LORA_ALPHA       = 32
LEARNING_RATE    = 3e-4
print('Config OK')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, TaskType, get_peft_model

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load in fp32 — LoRA adapters need fp32 weights to train correctly with fp16 trainer
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
)
if torch.cuda.is_available():
    model = model.cuda()

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q', 'v'],
    lora_dropout=0.05,
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def build_dataset(records):
    inputs  = [r['input'].strip()  for r in records if r.get('input')  and r.get('output')]
    outputs = [r['output'].strip() for r in records if r.get('input')  and r.get('output')]

    def tokenize(batch):
        model_inputs = tokenizer(
            batch['input'], max_length=MAX_INPUT_LEN, truncation=True, padding='max_length'
        )
        labels = tokenizer(
            text_target=batch['output'], max_length=MAX_TARGET_LEN, truncation=True, padding='max_length'
        )
        label_ids = [
            [(l if l != tokenizer.pad_token_id else -100) for l in lbl]
            for lbl in labels['input_ids']
        ]
        model_inputs['labels'] = label_ids
        return model_inputs

    raw = Dataset.from_dict({'input': inputs, 'output': outputs})
    return raw.map(tokenize, batched=True, remove_columns=['input', 'output'])

train_records = load_jsonl(TRAIN_FILE)
eval_records  = load_jsonl(EVAL_FILE)
print(f'Train: {len(train_records)}  Eval: {len(eval_records)}')

train_dataset = build_dataset(train_records)
eval_dataset  = build_dataset(eval_records)
print('Datasets tokenized')

In [ ]:
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    warmup_steps=100,
    learning_rate=LEARNING_RATE,
    fp16=False,
    logging_steps=50,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    report_to='none',
    dataloader_num_workers=2,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print('Starting training...')
trainer.train()

In [ ]:
final_dir = f'{OUTPUT_DIR}/final'
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f'Model saved to {final_dir}')
print('Download the "final" folder from Google Drive to use locally.')

In [ ]:
# Quick evaluation
from tqdm.notebook import tqdm

model.eval()
sample_records = eval_records[:100]
exact_matches = 0

for record in tqdm(sample_records):
    inp = record['input'].strip()
    target = record['output'].strip()

    inputs = tokenizer(inp, return_tensors='pt', max_length=MAX_INPUT_LEN, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_TARGET_LEN, num_beams=2)

    pred = tokenizer.decode(out[0], skip_special_tokens=True)
    if pred.strip() == target:
        exact_matches += 1

print(f'Exact Match on 100 eval samples: {exact_matches}%')